# CS781 Phase 1 - real training, two models in parallel (one per T4)
Set `GPU0_MODEL`, `GPU1_MODEL` and their micro-batch sizes (from the quick check), then **Save Version -> Save & Run All** (GPU T4 x2).
Each model trains on its own GPU for up to `MAX_HOURS` (stops cleanly, writes `final.pt` and `DONE`), or until all data has been seen.
Outputs: `/kaggle/working/run_<model>/` (latest.pt, best.pt, final.pt, metrics.jsonl, train_log.txt). Resume: attach this notebook's output as a dataset and set `RESUME_*`.

In [ ]:
import os, sys, json, re, glob, shutil, subprocess, time, threading
DATA = "/kaggle/input/datasets/mrinaldevnath/cs781-legal-data"
CODE_SRC = "/kaggle/input/datasets/mrinaldevnath/cs781-p1-train"
WORK = "/kaggle/working/cs781/train"

def find(root, name):
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(sorted(hits, key=len)[0])
DATA = find(DATA, "manifest.json")
CODE_SRC = find(CODE_SRC, "train.py")
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK)
print("data :", DATA, os.listdir(DATA))
print("code :", WORK, sorted(os.listdir(WORK)))
for f in ["sampler.py", "train.py", "generate.py", "tests.py", "configs/e.json"]:
    assert os.path.exists(f), f"MISSING {f} - re-upload the latest train_code.zip as the dataset"
assert "test_sampler" in open("tests.py").read(), "tests.py is an OLD version - re-upload the latest code"
assert "smoke_steps" in open("train.py").read(), "train.py is an OLD version - re-upload the latest code"
os.makedirs("/kaggle/working/logs", exist_ok=True)

def run_parallel(jobs, timeout=None):
    """jobs: list of (tag, shell_cmd, gpu_index_or_None). Runs them at the same time (one per GPU), streams
    output prefixed with the tag, saves /kaggle/working/logs/<tag>.txt, returns {tag: (exit_code, text)}."""
    procs, outs, lock = {}, {}, threading.Lock()
    def reader(tag, p):
        buf = []
        for line in p.stdout:
            buf.append(line)
            with lock:
                print(f"[{tag}] {line}", end="", flush=True)
        outs[tag] = buf
    threads = []
    for tag, cmd, gpu in jobs:
        env = dict(os.environ)
        if gpu is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        print(f"$ [{tag}] (GPU {gpu}) {cmd}", flush=True)
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env=env, start_new_session=True)
        procs[tag] = p
        t = threading.Thread(target=reader, args=(tag, p), daemon=True); t.start(); threads.append(t)
    timer = None
    if timeout:
        def kill():
            print("TIMEOUT - killing", flush=True)
            for p in procs.values():
                try: os.killpg(p.pid, 9)
                except Exception: pass
        timer = threading.Timer(timeout, kill); timer.start()
    for p in procs.values(): p.wait()
    for t in threads: t.join()
    if timer: timer.cancel()
    res = {}
    for tag, p in procs.items():
        txt = "".join(outs.get(tag, []))
        open(f"/kaggle/working/logs/{tag}.txt", "w").write(txt)
        res[tag] = (p.returncode, txt)
        print(f"[{tag}] exit {p.returncode}", flush=True)
    return res

In [ ]:
GPU0_MODEL, GPU0_MB = "b", 8        # models: a, b, c, d, e
GPU1_MODEL, GPU1_MB = "c", 8
MAX_HOURS = 11.0                    # per-session wall-clock limit measured from the start of THIS cell (Kaggle limit is 12 h)
GPU0_EXTRA = "--compile"           # e.g. "--compile --no_ckpt" if the speed-tuning notebook says so
GPU1_EXTRA = "--compile"
RESUME_GPU0 = None                  # e.g. "/kaggle/input/<old-output>/run_b_gpt_mha/latest.pt"
RESUME_GPU1 = None

In [ ]:
T0 = time.time()
# the setup cell above took a little time already; the train script measures --max_hours from ITS start, so subtract that
used = (time.time() - T0) / 3600
jobs = []
for gpu, (m, mb, resume, extra) in enumerate([(GPU0_MODEL, GPU0_MB, RESUME_GPU0, GPU0_EXTRA), (GPU1_MODEL, GPU1_MB, RESUME_GPU1, GPU1_EXTRA)]):
    if not m:
        continue
    name = json.load(open(f"configs/{m}.json"))["name"]
    cmd = (f"python train.py --config configs/{m}.json --data_root {DATA} --out /kaggle/working/run_{name} "
           f"--micro_batch {mb} --max_hours {MAX_HOURS} {extra}" + (f" --resume {resume}" if resume else ""))
    jobs.append((f"train_{m}", cmd, gpu))
res = run_parallel(jobs)
for tag, (rc, txt) in res.items():
    print(tag, "exit code", rc, "|", [l for l in txt.splitlines() if "DONE" in l][-1:])

In [ ]:
# loss curves from metrics.jsonl (runs after training; safe to ignore)
try:
    import pandas as pd, matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for f in glob.glob("/kaggle/working/run_*/metrics.jsonl"):
        df = pd.read_json(f, lines=True)
        n = os.path.basename(os.path.dirname(f))
        tr = df[df["loss"].notna()] if "loss" in df else df.iloc[0:0]
        ax[0].plot(tr["step"], tr["loss"], label=n)
        ev = df[df["eval"].notna()] if "eval" in df else df.iloc[0:0]
        ax[1].plot(ev["step"], [e["avg"] for e in ev["eval"]], marker="o", label=n)
    ax[0].set_title("train loss"); ax[1].set_title("val loss (shared metric, avg)")
    for a in ax: a.legend(); a.set_xlabel("step")
    plt.savefig("/kaggle/working/curves.png", dpi=110)
except Exception as e:
    print("plot skipped:", e)